In [13]:
import csv
import random
import math


# ==========================================
# 1️⃣ Load Dataset
# ==========================================
FILE_PATH = "Mall_Customers.csv"
INCOME_COLUMN = "Annual Income (k$)"
SPENDING_COLUMN = "Spending Score (1-100)"

customer_data = []

with open(FILE_PATH, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        income_value = float(row[INCOME_COLUMN])
        spending_value = float(row[SPENDING_COLUMN])
        customer_data.append([income_value, spending_value])


# ==========================================
# 2️⃣ Standardization (Manual Z-Score)
# ==========================================
def calculate_mean_and_std(values):
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    std_dev = math.sqrt(variance)
    return mean_value, std_dev


income_list = [point[0] for point in customer_data]
spending_list = [point[1] for point in customer_data]

income_mean, income_std = calculate_mean_and_std(income_list)
spending_mean, spending_std = calculate_mean_and_std(spending_list)

scaled_customers = [
    [
        (point[0] - income_mean) / income_std,
        (point[1] - spending_mean) / spending_std
    ]
    for point in customer_data
]


# ==========================================
# 3️⃣ Manual K-Means
# ==========================================
NUM_CLUSTERS = 5
MAX_ITERATIONS = 1000
random.seed(42)

# Randomly initialize centroids
centroid_list = random.sample(scaled_customers, NUM_CLUSTERS)


def squared_euclidean_distance(point, centroid):
    return (
        (point[0] - centroid[0]) ** 2 +
        (point[1] - centroid[1]) ** 2
    )


for iteration in range(MAX_ITERATIONS):

    # Step 1: Create empty clusters dictionary
    clusters_dict = {cluster_id: [] for cluster_id in range(NUM_CLUSTERS)}

    # Step 2: Assign each point to nearest centroid
    for point in scaled_customers:
        distances = [
            squared_euclidean_distance(point, centroid)
            for centroid in centroid_list
        ]
        closest_cluster = distances.index(min(distances))
        clusters_dict[closest_cluster].append(point)

    # Step 3: Recompute centroids
    new_centroid_list = []

    for cluster_id in range(NUM_CLUSTERS):
        cluster_points = clusters_dict[cluster_id]

        if cluster_points:
            mean_x = sum(p[0] for p in cluster_points) / len(cluster_points)
            mean_y = sum(p[1] for p in cluster_points) / len(cluster_points)
            new_centroid_list.append([mean_x, mean_y])
        else:
            # Reinitialize if cluster is empty
            new_centroid_list.append(random.choice(scaled_customers))

    # Step 4: Check convergence
    has_converged = True
    for i in range(NUM_CLUSTERS):
        movement = squared_euclidean_distance(
            centroid_list[i],
            new_centroid_list[i]
        )
        if movement > 1e-6:
            has_converged = False
            break

    centroid_list = new_centroid_list

    


# ==========================================
# 4️⃣ Final Cluster Assignment
# ==========================================
final_cluster_labels = []

for point in scaled_customers:
    distances = [
        squared_euclidean_distance(point, centroid)
        for centroid in centroid_list
    ]
    closest_cluster = distances.index(min(distances))
    final_cluster_labels.append(closest_cluster)


# ==========================================
# 5️⃣ Display Results (Original Scale)
# ==========================================
for cluster_id in range(NUM_CLUSTERS):

    original_points = [
        customer_data[index]
        for index, label in enumerate(final_cluster_labels)
        if label == cluster_id
    ]

    # Convert centroid back to original scale
    original_income = centroid_list[cluster_id][0] * income_std + income_mean
    original_spending = centroid_list[cluster_id][1] * spending_std + spending_mean

    print(f"\nCluster {cluster_id}")
    print("Size:", len(original_points))
    print("Centroid (Original Scale):",
          [round(original_income, 2), round(original_spending, 2)])
    print("Sample Points:", original_points[:10])


Cluster 0
Size: 36
Centroid (Original Scale): [83.11, 82.42]
Sample Points: [[69.0, 91.0], [70.0, 77.0], [71.0, 95.0], [71.0, 75.0], [71.0, 75.0], [72.0, 71.0], [73.0, 88.0], [73.0, 73.0], [74.0, 72.0], [75.0, 93.0]]

Cluster 1
Size: 34
Centroid (Original Scale): [83.06, 17.85]
Sample Points: [[70.0, 29.0], [71.0, 35.0], [71.0, 11.0], [71.0, 9.0], [72.0, 34.0], [73.0, 5.0], [73.0, 7.0], [74.0, 10.0], [75.0, 5.0], [76.0, 40.0]]

Cluster 2
Size: 23
Centroid (Original Scale): [26.3, 20.91]
Sample Points: [[15.0, 39.0], [16.0, 6.0], [17.0, 40.0], [18.0, 6.0], [19.0, 3.0], [19.0, 14.0], [20.0, 15.0], [20.0, 13.0], [21.0, 35.0], [23.0, 29.0]]

Cluster 3
Size: 7
Centroid (Original Scale): [123.57, 48.14]
Sample Points: [[99.0, 39.0], [120.0, 16.0], [120.0, 79.0], [126.0, 28.0], [126.0, 74.0], [137.0, 18.0], [137.0, 83.0]]

Cluster 4
Size: 100
Centroid (Original Scale): [48.26, 56.48]
Sample Points: [[15.0, 81.0], [16.0, 77.0], [17.0, 76.0], [18.0, 94.0], [19.0, 72.0], [19.0, 99.0], [20.0, 77